In [12]:
import pandas as pd
import numpy as np
import time
import os
import random
from pathlib import Path
try:
    import torch
except ModuleNotFoundError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
    import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

In [13]:
warnings.filterwarnings('ignore')

In [14]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [15]:
print("1. Loading 'bleaching_model_ready.csv'...")
DATA_PATH = Path("../data/bleaching_model_ready.csv")
ARTIFACT_DIR = Path("../model/custom_residual_mlp")
FINAL_ARTIFACT_PATH = ARTIFACT_DIR / "custom_residual_mlp_artifact.pt"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)

X = df.drop(columns=["Percent_Bleaching"])
y = df["Percent_Bleaching"].values.astype(np.float32)
y_log = np.log1p(y).astype(np.float32)  # Train on log-transformed target

# Build spatial groups for GroupKFold using latitude and longitude.
groups = df.groupby(['Latitude_Degrees', 'Longitude_Degrees']).ngroup().values

# Convert features to a NumPy array after defining groups.
feature_names = X.columns.tolist()
X = X.values.astype(np.float32)

print(f"   -> Rows: {X.shape[0]:,} | Features: {X.shape[1]} | Spatial groups: {len(np.unique(groups)):,}")

1. Loading 'bleaching_model_ready.csv'...
   -> Rows: 34,515 | Features: 61 | Spatial groups: 10,826


In [16]:
class CoralDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [17]:
# Change 1: use Sigmoid instead of Softmax in the attention gate.
class TabularAttention(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.Tanh(),
            nn.Linear(in_features // 2, in_features),
            nn.Sigmoid()  # Allow each feature to be weighted independently
        )
        
    def forward(self, x):
        attn_weights = self.attention(x)
        return x * attn_weights

# Change 2: use LayerNorm instead of BatchNorm1d.
class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)  # Replaces BatchNorm
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)  # Replaces BatchNorm
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        residual = x  
        out = F.relu(self.ln1(self.fc1(x)))
        out = self.dropout(out)
        out = self.ln2(self.fc2(out))
        out += residual  # Skip Connection
        out = F.relu(out)
        return out

class CoralResidualMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_blocks=3, dropout=0.2):
        super().__init__()
        self.attention = TabularAttention(input_dim)
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        x = self.attention(x)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.output_head(x)

In [18]:
def train_one_fold(fold, X_train, y_train_log, X_val, y_val_log, y_val_true, device):
    print(f"\n--- Starting Fold {fold+1} ---")
    
    # Fit scaling on the training split only to avoid leakage.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # DataLoaders
    batch_size = 256
    train_loader = DataLoader(CoralDataset(X_train_scaled, y_train_log), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(CoralDataset(X_val_scaled, y_val_log), batch_size=batch_size, shuffle=False)
    
    # Model setup
    model = CoralResidualMLP(input_dim=X.shape[1], hidden_dim=128, num_blocks=3, dropout=0.3).to(device)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    epochs = 50
    best_val_mae_real = float('inf')
    patience = 8
    patience_counter = 0
    
    for epoch in range(epochs):
        # -- Train --
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # -- Validation on the original target scale --
        model.eval()
        val_preds_log = []
        with torch.no_grad():
            for inputs, _ in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                val_preds_log.extend(outputs.cpu().numpy().flatten())
                
        # Convert predictions back to the 0-100 bleaching scale.
        val_preds_real = np.expm1(val_preds_log)
        val_preds_real = np.clip(val_preds_real, 0, 100)
        
        # Compute metrics on the real target scale.
        val_mae_real = mean_absolute_error(y_val_true, val_preds_real)
        val_rmse_real = np.sqrt(mean_squared_error(y_val_true, val_preds_real))
        
        # Early stopping is based on the real-scale MAE.
        if val_mae_real < best_val_mae_real:
            best_val_mae_real = val_mae_real
            best_rmse = val_rmse_real
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1} (best val MAE: {best_val_mae_real:.2f}%)")
            break
            
    return best_val_mae_real, best_rmse

In [19]:
print("\n2. Running GroupKFold (3 folds)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   -> Device: {device}")

gkf = GroupKFold(n_splits=3)
fold_maes = []
fold_rmses = []

start_total_time = time.time()

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_log, groups=groups)):
    X_train, y_train_log = X[train_idx], y_log[train_idx]
    X_val, y_val_log = X[val_idx], y_log[val_idx]
    y_val_true = y[val_idx]
    
    best_mae, best_rmse = train_one_fold(fold, X_train, y_train_log, X_val, y_val_log, y_val_true, device)
    
    fold_maes.append(best_mae)
    fold_rmses.append(best_rmse)
    
    print(f"   Done fold {fold+1} | RMSE: {best_rmse:.2f}% | MAE: {best_mae:.2f}%")

print("\n" + "="*60)
print(" MLP evaluation summary (GroupKFold)")
print("="*60)
print(f"    Average RMSE : {np.mean(fold_rmses):.2f} +/- {np.std(fold_rmses):.2f}%")
print(f"    Average MAE  : {np.mean(fold_maes):.2f} +/- {np.std(fold_maes):.2f}%")
print(f"    Total time   : {(time.time() - start_total_time)/60:.1f} minutes")
print("="*60)


2. Running GroupKFold (3 folds)...
   -> Device: cpu

--- Starting Fold 1 ---
    Early stopping at epoch 34 (best val MAE: 6.04%)
   Done fold 1 | RMSE: 14.00% | MAE: 6.04%

--- Starting Fold 2 ---
    Early stopping at epoch 23 (best val MAE: 6.49%)
   Done fold 2 | RMSE: 14.68% | MAE: 6.49%

--- Starting Fold 3 ---
    Early stopping at epoch 27 (best val MAE: 5.86%)
   Done fold 3 | RMSE: 13.15% | MAE: 5.86%

 MLP evaluation summary (GroupKFold)
    Average RMSE : 13.94 +/- 0.63%
    Average MAE  : 6.13 +/- 0.26%
    Total time   : 1.8 minutes


In [20]:
# 1. Fit the scaler on the full dataset.
final_scaler = StandardScaler()
X_full_scaled = final_scaler.fit_transform(X)  # Reuse X and y_log prepared above

# 2. Build the config needed for inference.
config = {
    "feature_names": feature_names,
    "input_dim": X.shape[1],
    "hidden_dim": 128,
    "num_blocks": 3,
    "dropout": 0.3,
}

# 3. Train the final model on the full dataset.
full_dataset = CoralDataset(X_full_scaled, y_log)
full_loader = DataLoader(full_dataset, batch_size=256, shuffle=True)

final_model = CoralResidualMLP(input_dim=X.shape[1], hidden_dim=128, num_blocks=3, dropout=0.3).to(device)
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=0.001, weight_decay=1e-4)

final_model.train()
for epoch in range(40):
    for inputs, targets in full_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        optimizer.step()

# 4. Save one deployment-ready artifact with model, scaler, and config.
artifact = {
    "model_state_dict": final_model.state_dict(),
    "scaler": final_scaler,
    "config": config,
    "metrics": {
        "cv_rmse_mean": float(np.mean(fold_rmses)),
        "cv_rmse_std": float(np.std(fold_rmses)),
        "cv_mae_mean": float(np.mean(fold_maes)),
        "cv_mae_std": float(np.std(fold_maes)),
    },
}
torch.save(artifact, FINAL_ARTIFACT_PATH)
print(f"    Saved single artifact to {FINAL_ARTIFACT_PATH}")

    Saved single artifact to ..\model\custom_residual_mlp\custom_residual_mlp_artifact.pt
